# Render + Eval
Render test poses (always with a lossless `--png_dir` archive), then score against the
eval split. `ckpt_latest.pt` works for crashed runs.

In [ ]:
# Run from the repo root; adjust if the notebook lives elsewhere
import os, sys
REPO = os.path.abspath(".")
assert os.path.isdir(os.path.join(REPO, "original")), "run this notebook from the repo root"
sys.path.insert(0, os.path.join(REPO, "original"))
DATA_ROOT = os.environ.get("DATA_ROOT", "dataset/raw")   # <-- EDIT: where the competition data lives
RUNS = os.environ.get("RUNS_ROOT", "runs")
os.makedirs(RUNS, exist_ok=True)
print("repo:", REPO, "| data:", DATA_ROOT)

In [ ]:
SCENE = os.path.join(DATA_ROOT, "SCENE_NAME")   # <-- EDIT
NAME = "SCENE_NAME_probe1"                        # <-- EDIT
CKPT = f"{RUNS}/{NAME}/ckpt.pt"
if not os.path.exists(CKPT): CKPT = f"{RUNS}/{NAME}/ckpt_latest.pt"
print("using", CKPT)

In [ ]:
!python original/render_gsplat.py --ckpt {CKPT} --csv {SCENE}/test/test_poses.csv \
  --out {RUNS}/{NAME}/test_render --png_dir {RUNS}/{NAME}/test_png 2>&1 | tail -3

## Eval split scoring (no test GT needed — isolated every-k holdout of TRAIN)

In [ ]:
# one-time per scene: build the split, retrain on train_sub, render eval_poses.csv, then:
#!python original/make_eval_split.py --scene {SCENE} --out {RUNS}/split_SCENE_NAME
!python original/eval_score.py --render_dir {RUNS}/{NAME}/eval_png --gt_dir {RUNS}/split_SCENE_NAME/eval_gt --tag {NAME}